### SK쉴더스 개발6기 홍서연
| First Assignment - 웹스크래핑 연습문제1
* 선택사항인 문제까지 모두 구현 완료하였습니다.

In [ ]:
# 1-1. Daum 뉴스기사 제목 스크래핑하기
import requests
from bs4 import BeautifulSoup

url = 'https://news.daum.net/economy'
headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/126.0 Safari/537.36'
    )
}

response = requests.get(url, headers=headers, timeout=10)
response.encoding = 'utf-8'

print(url)
print(type(response))
print(response.status_code)

soup = BeautifulSoup(response.text, 'html.parser')
li_tags = soup.select('ul.list_newsheadline2 li')

if not li_tags:
    li_tags = soup.select('ul.list_newsheadline li, li.item_news, div.item_news, a.link_txt')

print(type(li_tags), len(li_tags))

for li_tag in li_tags:
    if getattr(li_tag, 'name', '') == 'a':
        a_tag = li_tag
        title_tag = li_tag
    else:
        a_tag = li_tag.find('a', href=True)
        title_tag = (
            li_tag.select_one('div.cont_thumb strong.tit_txt')
            or li_tag.select_one('strong.tit_txt')
            or li_tag.select_one('.tit_txt')
            or a_tag
        )

    link = a_tag['href']
    title = title_tag.get_text(' ', strip=True)

    if title:
        print(link)
        print(title)

In [ ]:
# 1-2. Daum 뉴스기사 제목 스크래핑하기 코드를 섹션별 함수로 구현하기
import requests
from bs4 import BeautifulSoup

section_dict = {
    '기후/환경': 'climate',
    '사회': 'society',
    '경제': 'economy',
    '정치': 'politics',
    '국제': 'world',
    '문화': 'culture',
    '생활': 'life',
    'IT/과학': 'tech',
    '인물': 'people',
}

headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/126.0 Safari/537.36'
    )
}


def print_news(section_name):
    if section_name not in section_dict:
        print(f'지원하지 않는 섹션입니다: {section_name}')
        print('사용 가능한 섹션:', ', '.join(section_dict.keys()))
        return

    section = section_dict[section_name]
    url = f'https://news.daum.net/{section}'
    response = requests.get(url, headers=headers, timeout=10)
    response.encoding = 'utf-8'

    print(f'======> {url} {section_name} 뉴스 <======')

    soup = BeautifulSoup(response.text, 'html.parser')
    li_tags = soup.select('ul.list_newsheadline2 li')

    if not li_tags:
        li_tags = soup.select('ul.list_newsheadline li, li.item_news, div.item_news, a.link_txt')

    for li_tag in li_tags:
        if getattr(li_tag, 'name', '') == 'a':
            a_tag = li_tag
            title_tag = li_tag
        else:
            a_tag = li_tag.find('a', href=True)
            title_tag = (
                li_tag.select_one('div.cont_thumb strong.tit_txt')
                or li_tag.select_one('strong.tit_txt')
                or li_tag.select_one('.tit_txt')
                or a_tag
            )

        link = a_tag['href']
        title = title_tag.get_text(' ', strip=True)

        if title:
            print(link)
            print(title)


print_news('경제')
print_news('사회')

In [ ]:
# 2-1. Nate 뉴스기사 제목 스크래핑하기 (선택)
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from IPython.display import Image, display

nate_section_dict = {
    '최신뉴스': 'n0100',
    '정치': 'n0200',
    '경제': 'n0300',
    '사회': 'n0400',
    '세계': 'n0500',
    'IT/과학': 'n0600',
}

headers = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/126.0 Safari/537.36'
    )
}


def print_nate_news(section_name, limit=5):
    if section_name not in nate_section_dict:
        print(f'지원하지 않는 섹션입니다: {section_name}')
        print('사용 가능한 섹션:', ', '.join(nate_section_dict.keys()))
        return

    mid = nate_section_dict[section_name]
    url = f'https://news.nate.com/recent?mid={mid}'
    response = requests.get(url, headers=headers, timeout=10)
    response.encoding = response.apparent_encoding or 'utf-8'

    soup = BeautifulSoup(response.text, 'html.parser')
    candidates = soup.select('div.mduSubjectList li, div.postSubjectContent, li')

    news_items = []
    seen_links = set()

    for item in candidates:
        a_tag = item.select_one('a[href*="/view/"]') or item.find('a', href=True)
        if not a_tag:
            continue

        link = urljoin(url, a_tag['href'])
        if '/view/' not in link or link in seen_links:
            continue

        title_tag = (
            item.select_one('strong.tit')
            or item.select_one('h2.tit')
            or item.select_one('span.tit')
            or item.select_one('.tit')
            or a_tag
        )
        title = title_tag.get_text(' ', strip=True)
        if not title:
            continue

        img_tag = item.find('img')
        img_url = None
        if img_tag:
            src = img_tag.get('data-src') or img_tag.get('src')
            if src:
                img_url = urljoin(url, src.strip())

        news_items.append({'title': title, 'link': link, 'image': img_url})
        seen_links.add(link)

        if len(news_items) >= limit:
            break

    print(f'======> {url} {section_name} 뉴스 <======')

    for item in news_items:
        print('기사제목:', item['title'])
        print('기사링크:', item['link'])

        if item['image']:
            print('Image:', item['image'])
            display(Image(url=item['image'], width=160))
        else:
            print('Image: 이미지 없음')

        print()


for section_name in nate_section_dict:
    print_nate_news(section_name)

In [ ]:
# 2-2. 하나의 네이버 웹툰과 1개의 회차에 대한 Image 다운로드 하기
import re
import requests
from pathlib import Path
from urllib.parse import urljoin, urlparse
from bs4 import BeautifulSoup
from IPython.display import Image, display


def clean_dir_name(value):
    return re.sub(r'[\\/:*?"<>|]', '_', str(value)).strip()


def download_one_episode(title, no, url):
    title_dir = clean_dir_name(title)
    save_dir = Path('img') / title_dir / str(no)
    save_dir.mkdir(parents=True, exist_ok=True)

    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/126.0 Safari/537.36'
        ),
        'Referer': url,
    }

    response = requests.get(url, headers=headers, timeout=10)
    response.encoding = 'utf-8'
    response.raise_for_status()

    soup = BeautifulSoup(response.text, 'html.parser')
    viewer = (
        soup.select_one('div.wt_viewer')
        or soup.select_one('#comic_view_area')
        or soup.select_one('div.viewer_img')
        or soup
    )

    image_urls = []
    for img_tag in viewer.select('img'):
        src = img_tag.get('data-src') or img_tag.get('src')
        if not src:
            continue

        img_url = urljoin(url, src.strip())

        if 'image-comic.pstatic.net' not in img_url and 'comic.naver.com' not in img_url:
            continue

        if img_url not in image_urls:
            image_urls.append(img_url)

    if not image_urls:
        raise ValueError('다운로드할 웹툰 이미지를 찾지 못했습니다.')

    saved_files = []
    for index, img_url in enumerate(image_urls, start=1):
        extension = Path(urlparse(img_url).path).suffix.lower()
        if extension not in ['.jpg', '.jpeg', '.png', '.gif', '.webp']:
            extension = '.jpg'

        file_path = save_dir / f'{index:03d}{extension}'
        image_response = requests.get(img_url, headers=headers, timeout=10)
        image_response.raise_for_status()

        with open(file_path, 'wb') as file:
            file.write(image_response.content)

        saved_files.append(file_path)
        print(file_path)

    print(f'총 {len(saved_files)}개의 이미지를 {save_dir}에 다운로드했습니다.')

    for file_path in saved_files[:3]:
        display(Image(filename=str(file_path), width=240))

    return saved_files


download_one_episode(
    '일렉시드',
    341,
    'https://comic.naver.com/webtoon/detail?titleId=717481&no=341&week=wed',
)